# Tier 4.1 — Agentic CRAG Experiments (Azure GPU) — Primary v2 Run

**BSARD RAG Thesis | RQ1 | Evaluator: LLaMA 3.1 8B collective judgment**

## This run: BM25 variant on test split, collective LLM evaluator

One CRAG experiment on the **test split** (222 questions), using a single collective
LLM sufficiency call per iteration with progressive criterion relaxation.

The aligned-mode run (top_k=20) is preserved in §Aligned Ablation below for reference.

| Experiment | Backbone | Purpose |
|---|---|---|
| `crag_bm25_test_v2` | `bm25_tuned_k11.5_b0.25` | CRAG on sparse backbone |

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC4as_T4_v3`
2. **Set Cell 1** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. Run cells top to bottom

### How to generate the Container SAS URL
Azure Portal → Storage Accounts → *your account* → Containers → `bsard-data`
→ `...` → **Generate SAS** → Permissions: **Read + List** → Expiry: 1 year
→ Generate → copy the **Blob SAS URL** (full `https://...` URL, not just the token)

## Expected execution times (T4 GPU)

Three-scenario collective model (`eval_k=20`, single LLM call per iteration):
- Scenario A (Correct first-pass, ~50%): retrieval + 1 collective eval call
- Scenario B (one rewrite, ~42%): +2nd retrieval + rewrite + 2nd collective eval
- Scenario C (two rewrites, ~8%): +3rd retrieval + aspect extract + focused rewrite + 3rd collective eval

| Phase | Expected time |
|---|---|
| Setup (Cells 0–7) | ~20 min |
| BM25 test run (`crag_bm25_test_v2`) | ~64 min (actual) |
| **Total** | **~67 min (actual)** |

## Resuming after interruption
Each experiment writes its JSON immediately and skips if the file exists.
Re-run from Cell 0 — completed experiments are skipped automatically.
Commit intermediate results before stopping the VM.

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)


In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

# ── Install Ollama if not present ─────────────────────────────────────────────
if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

# ── Start server (if not already running) ─────────────────────────────────────
if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',           # use all GPU layers
            'OLLAMA_FLASH_ATTENTION': '1',    # faster on A10/T4
            'OLLAMA_HOST': '0.0.0.0:11434',
            'OLLAMA_NUM_CTX': '20000',        # context window for eval_k=10–50 collective prompts
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)

    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

# ── Verify ─────────────────────────────────────────────────────────────────────
resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')

In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'], check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path

OUTPUT_DIR = Path(REPO_DIR) / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BM25-only: no embedding files needed
downloads = {
    'bsard_articles_dedup.parquet': OUTPUT_DIR,
    'bsard_corpus.db':              OUTPUT_DIR,
}

client = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

for blob_name, dest_dir in downloads.items():
    dest_path = Path(dest_dir) / blob_name
    if dest_path.exists():
        print(f'  Already exists: {blob_name} ({dest_path.stat().st_size / 1e6:.1f} MB)')
        continue
    print(f'  Downloading {blob_name} ...', end='', flush=True)
    with open(dest_path, 'wb') as f:
        client.get_blob_client(blob_name).download_blob().readinto(f)
    print(f' done ({dest_path.stat().st_size / 1e6:.1f} MB)')

print('\nAll data files ready.')


In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    # Re-embed token in remote URL so pull works on a fresh VM session
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys, os

cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
cuda_ver = 'cu118' if 'release 11' in cuda_out else 'cu121'
print(f'CUDA detected → torch variant: {cuda_ver}')

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
      '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}'],
     'torch'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
    # tf-keras: transformers tries to import Keras 3 (which breaks) — this shim fixes it
    ([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'],
     'tf-keras'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'],
     'spacy'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package from the RQ3 repo (required by evaluation/runner.py)
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

In [ ]:
# ── Cell 6: Install spaCy French model ───────────────────────────────────────
# Uses direct pip wheel URL to avoid `spacy download` 404 errors on GitHub releases.
import subprocess, sys
import spacy as _spacy

_sv   = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    # Fallback: try major.minor.0 in case patch version differs
    _sv2  = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'spaCy fr_core_news_lg install failed:\n{r.stderr[-400:]}')

import spacy
nlp = spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# ── Cell 7: Pre-flight checks + LLM latency benchmark ────────────────────────
import os, sys, time
import requests as _req
from pathlib import Path

os.chdir(REPO_DIR)

# ── File checks ───────────────────────────────────────────────────────────────
for p in [
    Path('output/bsard_articles_dedup.parquet'),
    Path('output/bsard_corpus.db'),
    Path('evaluation/data/fewshot_examples.json'),
]:
    print(f'  {"OK     " if p.exists() else "MISSING"}  {p}')

# ── Ollama alive ──────────────────────────────────────────────────────────────
try:
    r      = _req.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'\nOllama alive. Models: {models}')
except Exception as e:
    raise RuntimeError(f'Ollama not reachable: {e}')

# ── LLM latency benchmark — collective prompt (eval_k=10 simulation) ──────────
# Simulates the actual eval call: ~3,200 token input (10 articles × 300 tokens)
SAMPLE_ARTICLES = '\n\n'.join(
    f'--- Article {i+1} ---\n'
    'Les travailleurs salariés ont droit à un congé parental pour s\'occuper d\'un enfant. '
    'Les conditions d\'octroi sont fixées par la convention collective de travail. '
    'La durée maximale est déterminée en fonction de l\'ancienneté du travailleur dans l\'entreprise.'
    for i in range(20)
)
COLLECTIVE_PROMPT = (
    'Question : Quelles sont les conditions pour obtenir un congé parental en Belgique ?\n\n'
    f'{SAMPLE_ARTICLES}\n\n'
    'Ces articles fournissent-ils collectivement une base juridique suffisante pour répondre '
    'à cette question de manière complète et précise ?\n'
    'Répondez par « Oui » ou « Non » en premier, suivi d\'une brève justification.\n\nRéponse :'
)

def llm_generate(prompt, max_tokens=80):
    t0   = time.perf_counter()
    resp = _req.post('http://localhost:11434/api/generate', json={
        'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
        'options': {'temperature': 0.0, 'num_predict': max_tokens},
    }, timeout=300)
    return resp.json()['response'].strip(), (time.perf_counter() - t0) * 1000

print('\nBenchmarking collective LLM call (eval_k=10 simulation, 3 calls)...')
lats = []
for i in range(3):
    resp, lat = llm_generate(COLLECTIVE_PROMPT, max_tokens=80)
    lats.append(lat)
    short_resp = resp[:50].replace('\n', ' ')
    print(f'  Call {i+1}: {short_resp!r:52s}  {lat:.0f} ms')

warm_lat = sum(lats[1:]) / 2  # discard cold-start (call 1), average calls 2–3
mean_lat = sum(lats) / len(lats)
print(f'\nWarm mean (calls 2–3): {warm_lat:.0f} ms  |  Overall mean: {mean_lat:.0f} ms')

if warm_lat < 10_000:
    print('GPU confirmed (fast).')
elif warm_lat < 60_000:
    print('MARGINAL — verify GPU in Cell 1.')
else:
    print('WARNING: likely on CPU. Re-check nvidia-smi.')

# ── Three-scenario time estimate (collective judgment, eval_k=10) ─────────────
_RETRIEVAL_MS   = 2_000
_REWRITE_STD_MS = 2_800
_REWRITE_ASP_MS = 10_000  # aspect extract + focused rewrite combined
n_test          = 222

p_a = 0.50  # Correct first-pass
p_b = 0.42  # one standard rewrite
p_c = 0.08  # two rewrites (aspect extract + focused)

lat_a = _RETRIEVAL_MS + warm_lat
lat_b = 2*_RETRIEVAL_MS + warm_lat + _REWRITE_STD_MS + warm_lat
lat_c = 3*_RETRIEVAL_MS + warm_lat + _REWRITE_STD_MS + warm_lat + _REWRITE_ASP_MS + warm_lat

mean_query_ms   = p_a*lat_a + p_b*lat_b + p_c*lat_c
est_min_variant = (n_test * mean_query_ms / 1_000) / 60.0

print(f'\n--- Estimated times per variant at warm_lat={warm_lat:.0f} ms/collective-call ---')
print(f'  Scenario A ({p_a:.0%} Correct 1st-pass): {lat_a/1000:.1f} s/query')
print(f'  Scenario B ({p_b:.0%} 1-rewrite)       : {lat_b/1000:.1f} s/query')
print(f'  Scenario C ({p_c:.0%} 2-rewrite)       : {lat_c/1000:.1f} s/query')
print(f'  Weighted mean                           : {mean_query_ms/1000:.1f} s/query')
print(f'  Estimate per variant (222q)             : ~{est_min_variant:.0f} min')


---
## Primary v2 Run — BM25 Variant, Test Split

Collective LLM evaluator (`eval_k=20`, single sufficiency call per iteration).
Progressive criterion relaxation: iteration 0 = strict Variant B, iterations 1+ = lenient Variant D.
BM25 backbone aligned with T4.0 first stage (`bm25_tuned_k11.5_b0.25`).

In [ ]:
# ── Cell: eval_k configuration ───────────────────────────────────────────────
# eval_k=20 is locked at the aligned-run value (95.9% Correct first-pass confirmed).
# Override below if needed for ablations.
#
# Auto eval_k probe: if EVAL_K is set to None, the script will probe val questions
# with the BM25 backbone to find the largest eval_k that fits within MAX_MINUTES.

EVAL_K         = 20    # docs judged per collective LLM call (set None for auto-probe)
MAX_MINUTES    = 90    # per-variant time budget for auto eval_k probe (ignored if EVAL_K set)
MAX_ITERATIONS = 6     # CRAG loop hard limit; 0.0% limit-terminated in test run
MAX_ART_TOKENS = 400   # per-article token budget → ~8,000 total for eval_k=20
BACKBONE_TOP_K = 100   # retrieve this many before evaluate + finalize

print(f'eval_k         = {EVAL_K}  (None = auto-probe)')
print(f'max_iterations = {MAX_ITERATIONS}')
print(f'max_art_tokens = {MAX_ART_TOKENS}')
print(f'backbone_top_k = {BACKBONE_TOP_K}')
print(f'max_minutes    = {MAX_MINUTES}  (auto-probe budget per variant)')

In [ ]:
# ── Cell: max_iterations sensitivity — execution time vs limit-termination ────
# Depends on warm_lat, _RETRIEVAL_MS, _REWRITE_STD_MS, _REWRITE_ASP_MS from Cell 7.
# This cell makes NO LLM calls — pure arithmetic from the warm_lat benchmark.
#
# Purpose:
#   1. Show estimated total runtime + limit-termination rate for max_iterations 1–4.
#   2. Given TIME_BUDGET_MIN, compute the largest n_questions that fits within budget
#      at the current MAX_ITERATIONS — use that as the smoke-test subset size.
#   3. Emit a final recommendation so you can set MAX_ITERATIONS + N_SMOKE_TEST
#      before starting the run.

#TIME_BUDGET_MIN = 15   # ← set your wall-clock budget here (minutes)
N_TEST_FULL     = 222  # full test split

_REWRITE_FOCUS_MS    = _REWRITE_STD_MS   # iter 2+ focused rewrite ≈ 1 LLM call (same as std)

# Conditional pass rates (derived from Cell 7 scenario mix: p_a=0.50, p_b=0.42, p_c=0.08)
P_CORRECT_ITER0      = 0.50   # P(Correct | iteration 0)
P_CORRECT_ITER1PLUS  = 0.84   # P(Correct | iteration k≥1, reached)  [= 0.42 / 0.50]

ITER_BASE_MS = _RETRIEVAL_MS + warm_lat  # retrieve + one collective eval LLM call (ms)

# ── Model helpers ─────────────────────────────────────────────────────────────
def _rewrite_ms(iter_idx):
    """Rewrite LLM cost fired AFTER a Non judgment at `iter_idx`."""
    if iter_idx == 0:   return _REWRITE_STD_MS    # standard statutory rewrite (1 call)
    elif iter_idx == 1: return _REWRITE_ASP_MS    # aspect extract + focused rewrite (2 calls)
    else:               return _REWRITE_FOCUS_MS  # focused rewrite only (1 call)

def _query_ms(depth):
    """Wall-clock ms for a query that runs (depth+1) retrieve+eval cycles."""
    return (depth + 1) * ITER_BASE_MS + sum(_rewrite_ms(i) for i in range(depth))

def _frac(k, max_iter):
    """
    Fraction of queries terminating at depth k (iterations 0..k executed).
    k < max_iter : resolved Correct.   k == max_iter : hit iteration limit.
    """
    if k == 0:
        return P_CORRECT_ITER0
    elif k < max_iter:
        return (1 - P_CORRECT_ITER0) * (1 - P_CORRECT_ITER1PLUS)**(k - 1) * P_CORRECT_ITER1PLUS
    else:
        return (1 - P_CORRECT_ITER0) * (1 - P_CORRECT_ITER1PLUS)**(max_iter - 1)

def _mean_ms(max_iter):
    """Expected wall-clock ms per query at a given max_iter."""
    return sum(_frac(k, max_iter) * _query_ms(k) for k in range(max_iter + 1))

def _simulate(max_iter, n):
    mean   = _mean_ms(max_iter)
    fracs  = {k: _frac(k, max_iter) for k in range(max_iter + 1)}
    return {
        'mean_s':      mean / 1000,
        'total_min':   (n * mean / 1000) / 60,
        'frac_c0':     fracs[0],
        'frac_c1plus': sum(fracs[k] for k in range(1, max_iter)),
        'frac_limit':  fracs[max_iter],
    }

# def _max_questions(max_iter, budget_min):
#     """Largest integer n such that estimated total runtime ≤ budget_min minutes."""
#     return int((budget_min * 60 * 1000) / _mean_ms(max_iter))

# ── Header ────────────────────────────────────────────────────────────────────
print(f'Warm LLM latency  : {warm_lat:.0f} ms/call')
print(f'ITER_BASE_MS      : {ITER_BASE_MS:.0f} ms  (retrieve + collective eval)')
print(f'Rewrite costs     : std={_REWRITE_STD_MS} ms  |  '
      f'aspect+focused={_REWRITE_ASP_MS} ms  |  focused={_REWRITE_FOCUS_MS} ms')
print(f'P(Correct iter 0) : {P_CORRECT_ITER0:.0%}  '
      f'|  P(Correct iter 1+|reached) : {P_CORRECT_ITER1PLUS:.0%}')
print(f'Full test split   : {N_TEST_FULL} questions')
print()

# ── Sensitivity table (full split) ────────────────────────────────────────────
print(f'{"max_iter":<10}{"mean s/q":>10}{"total (min)":>13}{"Correct@0":>11}'
      f'{"Correct@1+":>12}{"hit limit":>11}  notes')
print('─' * 82)
for mi in [1, 2, 3, 4]:
    s      = _simulate(mi, N_TEST_FULL)
    notes  = []
    if mi == MAX_ITERATIONS:         notes.append('← current')
    if s['frac_limit'] > 0.30:       notes.append('*** >30% limit')
    elif s['frac_limit'] < 0.05:     notes.append('< 5% limit (marginal gain)')
    #fits   = '✓' if s['total_min'] <= TIME_BUDGET_MIN else '✗'
    print(f'  {mi:<8}{s["mean_s"]:>10.1f}{s["total_min"]:>11.0f} min'
        f'{s["frac_c0"]:>11.1%}{s["frac_c1plus"]:>11.1%}{s["frac_limit"]:>10.1%}'
        f'  {"  ".join(notes)}')


# # ── Budget analysis ───────────────────────────────────────────────────────────
# print()
# print(f'── Budget analysis: {TIME_BUDGET_MIN} min cap ──────────────────────────────────')

# s_full = _simulate(MAX_ITERATIONS, N_TEST_FULL)
# n_budget = _max_questions(MAX_ITERATIONS, TIME_BUDGET_MIN)
# n_smoke  = min(n_budget, N_TEST_FULL)

# if s_full['total_min'] <= TIME_BUDGET_MIN:
#     print(f'  Full run ({N_TEST_FULL} questions) fits within {TIME_BUDGET_MIN} min '
#           f'at MAX_ITERATIONS={MAX_ITERATIONS}.')
#     print(f'  Estimated: {s_full["total_min"]:.0f} min  (margin: '
#           f'+{TIME_BUDGET_MIN - s_full["total_min"]:.0f} min)')
#     N_SMOKE_TEST = N_TEST_FULL
# else:
#     print(f'  Full run ({N_TEST_FULL} q) estimated at {s_full["total_min"]:.0f} min '
#           f'— exceeds {TIME_BUDGET_MIN} min budget.')
#     print(f'  At MAX_ITERATIONS={MAX_ITERATIONS}, budget fits: {n_smoke} questions.')
#     print(f'  → Use N_SMOKE_TEST = {n_smoke} for a ~{TIME_BUDGET_MIN} min validation run.')
#     N_SMOKE_TEST = n_smoke

# # ── Delta: cost of going from max_iter=2 to max_iter=3 ────────────────────────
# print()
# r2, r3 = _simulate(2, N_TEST_FULL), _simulate(3, N_TEST_FULL)
# delta_min = r3['total_min'] - r2['total_min']
# print(f'Δ max_iter 2→3 (full run) :  +{delta_min:.0f} min  |  '
#       f'limit-terminated {r2["frac_limit"]:.1%} → {r3["frac_limit"]:.1%} '
#       f'(−{r2["frac_limit"] - r3["frac_limit"]:.1%} queries)')

# ── Recommendation ────────────────────────────────────────────────────────────
print()
print('── Recommendation ──────────────────────────────────────────────────────────')
print(f'  Rule of thumb: smallest max_iter where hit-limit < 10%.')
print()
for mi, label in [(1, 'too restrictive (50% limit-terminated)'),
                  (2, 'default — recommended baseline'),
                  (3, 'conservative — use if >10% limit-terminated in post-run review'),
                  (4, 'rarely needed')]:
    s      = _simulate(mi, N_TEST_FULL)
    marker = '>>>' if mi == MAX_ITERATIONS else '   '
    fits   = f'{s["total_min"]:.0f} min' if s['total_min'] <= TIME_BUDGET_MIN else f'{s["total_min"]:.0f} min (over budget)'
    print(f'  {marker} max_iter={mi}: {s["frac_limit"]:.1%} limit-terminated  '
          f'| full run ≈ {fits}  ({label})')

print()
print(f'Current setting : MAX_ITERATIONS = {MAX_ITERATIONS}')
s_cur = _simulate(MAX_ITERATIONS, N_TEST_FULL)
if s_cur['frac_limit'] > 0.30:
    print(f'  WARNING: {s_cur["frac_limit"]:.1%} expected limit-termination '
          f'— increase MAX_ITERATIONS.')
else:
    print(f'  OK: {s_cur["frac_limit"]:.1%} expected limit-termination '
          f'(within <30% threshold).')

# if s_full['total_min'] > TIME_BUDGET_MIN:
#     print()
#     print(f'  For a ≤{TIME_BUDGET_MIN} min validation, set N_SMOKE_TEST = {N_SMOKE_TEST} '
#           f'in run_crag_experiments.py (--n-questions {N_SMOKE_TEST}).')
#     print(f'  Run the full {N_TEST_FULL}-question test after confirming the pipeline.')


In [ ]:
# ── Cell: BM25 test run — crag_bm25_test_v2 ──────────────────────────────────
# Backbone: bm25_tuned_k11.5_b0.25  (T4.0-aligned BM25: k1=1.5, b=0.25, lemmatize, text_only)
# Writes: output/results/agentic/CRAG/crag_bm25_test_v2.json
# Actual: ~64 min (eval_k=20, max_iterations=6)
import json, os, subprocess, sys, time
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT = RESULTS_DIR / 'crag_bm25_test_v2.json'

if OUT.exists():
    r10 = json.loads(OUT.read_text())['metrics'].get('Recall@10', 0)
    print(f'Already exists — skipping.  R@10={r10:.4f}')
else:
    pull = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True
    )
    print('git pull:', pull.stdout.strip() or pull.stderr.strip())

    print(f'Running CRAG BM25 test (~45-55 min, eval_k={EVAL_K})...')
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, 'scripts/evaluation/tier4/run_crag_experiments.py',
         '--split', 'test', '--variant', 'bm25',
         '--max-iterations', str(MAX_ITERATIONS),
         '--eval-k', str(EVAL_K),
         '--max-article-tokens', str(MAX_ART_TOKENS)],
        cwd=REPO_DIR, timeout=14400,
    )
    print(f'\nDone in {(time.time()-t0)/60:.1f} min (exit {result.returncode})')
    if result.returncode != 0:
        raise RuntimeError('BM25 experiment failed — check output above')


In [ ]:
# ── Cell: Review BM25 result ──────────────────────────────────────────────────
import json
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
path = RESULTS_DIR / 'crag_bm25_test_v2.json'

if not path.exists():
    print('crag_bm25_test_v2.json not found — run BM25 experiment first.')
else:
    r    = json.loads(path.read_text())
    m    = r.get('metrics', {})
    loop = r.get('crag_loop_stats', {})
    bd   = r.get('latency_breakdown_ms_mean', {})
    lim  = loop.get('fraction_queries_terminated_by_limit', 0)

    print(f'{"Metric":<25} {"Value":>10}')
    print('-' * 38)
    print(f'  {"R@10":<23} {m.get("Recall@10", 0):>10.4f}')
    print(f'  {"R@100":<23} {m.get("Recall@100", 0):>10.4f}')
    print(f'  {"MRR@10":<23} {m.get("MRR@10", 0):>10.4f}')
    print(f'  {"Mean latency (ms)":<23} {r.get("latency_ms_mean", 0):>10.0f}')
    print()
    print('Loop stats:')
    print(f'  Correct first-pass  : {loop.get("fraction_queries_correct_first_pass", 0):.1%}')
    print(f'  Rewritten           : {loop.get("fraction_queries_rewritten", 0):.1%}')
    print(f'  Used aspect rewrite : {loop.get("fraction_queries_used_aspect_rewrite", 0):.1%}')
    print(f'  Terminated by limit : {lim:.1%}')
    print()
    print('Latency breakdown (ms/query):')
    print(f'  retrieval={bd.get("retrieval", 0):.0f}  '
          f'evaluator_llm={bd.get("evaluator_llm", 0):.0f}  '
          f'rewrite_llm={bd.get("rewrite_llm", 0):.0f}')

    if lim > 0.30:
        print(f'\n*** WARNING: {lim:.0%} queries hit max_iterations limit.')
        print('   Consider increasing MAX_ITERATIONS to 3 before re-running.')

In [ ]:
# ── Cell: Per-iteration recall progression ────────────────────────────────────
# Shows how retrieval quality evolves across CRAG iterations for rewritten queries.
# Requires updated crag.py (retrieved_article_ids in trace) and run_crag_experiments.py.
import json
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
path = RESULTS_DIR / 'crag_bm25_test_v2.json'

if not path.exists():
    print('crag_bm25_test_v2.json not found — run BM25 experiment first.')
else:
    r   = json.loads(path.read_text())
    pit = r.get('per_iteration_recall', {})

    if not pit:
        print('per_iteration_recall not in result.')
        print('Re-run the experiment with the updated crag.py and run_crag_experiments.py.')
    else:
        print('── Per-iteration recall @ eval_k ────────────────────────────────')
        for k in sorted(k for k in pit if k.startswith('mean_recall_at_eval_k_iter_')):
            iter_idx = k.split('_')[-1]
            print(f'  Iteration {iter_idx} : {pit[k]:.4f}')
        print()

        n_rw = pit.get('n_queries_with_rewrites', 0)
        print(f'Queries with ≥1 rewrite : {n_rw}')
        if n_rw > 0:
            print(f'  Improved  after rewrite : '
                  f'{pit.get("fraction_improved_after_rewrite",  0):.1%}'
                  '  (rewrite found more relevant articles within eval_k)')
            print(f'  Unchanged after rewrite : '
                  f'{pit.get("fraction_unchanged_after_rewrite", 0):.1%}'
                  '  (same coverage)')
            print(f'  Regressed after rewrite : '
                  f'{pit.get("fraction_regressed_after_rewrite", 0):.1%}'
                  '  (fewer — best_seen prevents this affecting final output)')
            print(f'  Mean recall Δ           : '
                  f'{pit.get("mean_recall_delta_after_rewrite", 0):+.4f}')
        print()
        print('Note: recall is measured over the eval_k documents shown to the LLM,')
        print('not over the full top-100 candidate pool used for final ranking.')


In [ ]:
# ── Cell: Final results — BM25 v2 + BM25 baseline (Tier 1) ───────────────────
import json
from pathlib import Path

RESULTS_DIR = Path(f'{REPO_DIR}/output/results/agentic/CRAG')
BM25_BASE   = Path(f'{REPO_DIR}/output/results/sparse_retrieval/bm25_tuned_k11.5_b0.25_test.json')
BM25_PATH   = RESULTS_DIR / 'crag_bm25_test_v2.json'

print(f'{"Experiment":<50} {"R@10":>7} {"R@100":>7} {"MRR@10":>8} {"Lat(ms)":>9}')
print('=' * 85)

# BM25 baseline (Tier 1 anchor)
if BM25_BASE.exists():
    base = json.loads(BM25_BASE.read_text())
    m    = base['metrics']
    print(f'  {"bm25_tuned_k11.5_b0.25 (Tier 1 baseline)":<48}'
          f'{m.get("Recall@10",0):>7.4f}'
          f'{m.get("Recall@100",0):>7.4f}'
          f'{m.get("MRR@10",0):>8.4f}'
          f'{base.get("latency_ms_mean",0):>9.0f}')
else:
    print(f'  BM25 baseline not found at {BM25_BASE}')
print('-' * 85)

# BM25 CRAG v2 result
if BM25_PATH.exists():
    r    = json.loads(BM25_PATH.read_text())
    m    = r.get('metrics', {})
    loop = r.get('crag_loop_stats', {})
    lim  = loop.get('fraction_queries_terminated_by_limit', 0)
    print(f'  {"crag_bm25_test_v2":<48}'
          f'{m.get("Recall@10",0):>7.4f}'
          f'{m.get("Recall@100",0):>7.4f}'
          f'{m.get("MRR@10",0):>8.4f}'
          f'{r.get("latency_ms_mean",0):>9.0f}')
    if lim > 0.30:
        print(f'    *** WARNING: {lim:.0%} limit-terminated — consider increasing MAX_ITERATIONS to 3')
else:
    print(f'  {"crag_bm25_test_v2":<48}  (not yet run)')

print()

# Delta: crag_bm25 vs Tier 1 BM25 baseline
if BM25_PATH.exists() and BM25_BASE.exists():
    crag_m  = json.loads(BM25_PATH.read_text())['metrics']
    base_m  = json.loads(BM25_BASE.read_text())['metrics']
    delta   = crag_m.get('Recall@10', 0) - base_m.get('Recall@10', 0)
    print(f'Δ R@10 (crag_bm25_test_v2 − BM25 baseline): {delta:+.4f}')
    print('Same backbone, same corpus — delta = CRAG correction loop contribution over raw BM25.')

# Loop stats
if BM25_PATH.exists():
    r    = json.loads(BM25_PATH.read_text())
    loop = r.get('crag_loop_stats', {})
    bd   = r.get('latency_breakdown_ms_mean', {})
    print(f'\nLoop stats (crag_bm25_test_v2):')
    print(f'  Correct first-pass  : {loop.get("fraction_queries_correct_first_pass", 0):.1%}')
    print(f'  Rewritten           : {loop.get("fraction_queries_rewritten", 0):.1%}')
    print(f'  Used aspect rewrite : {loop.get("fraction_queries_used_aspect_rewrite", 0):.1%}')
    print(f'  Terminated by limit : {loop.get("fraction_queries_terminated_by_limit", 0):.1%}')
    print(f'\nLatency breakdown (ms/query):')
    print(f'  retrieval={bd.get("retrieval",0):.0f}  '
          f'evaluator_llm={bd.get("evaluator_llm",0):.0f}  '
          f'rewrite_llm={bd.get("rewrite_llm",0):.0f}')

In [ ]:
# ── Cell: Upload primary v2 results to Azure Blob Storage ─────────────────────
# output/ is gitignored — results are NOT committed to git.
# Upload to blob container so they sync to local output/ via blob download.
# The SAS URL must have Write permission.
from pathlib import Path
from azure.storage.blob import ContainerClient

RESULT_DIRS = {
    'results/agentic/CRAG': Path(f'{REPO_DIR}/output/results/agentic/CRAG'),
}

client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
uploaded = []

for blob_prefix, results_dir in RESULT_DIRS.items():
    if not results_dir.exists():
        print(f'  [{blob_prefix}] directory not found — skipping.')
        continue
    for json_file in sorted(results_dir.glob('*.json')):
        blob_name = f'{blob_prefix}/{json_file.name}'
        print(f'  Uploading {json_file.name} → {blob_name} ...', end='', flush=True)
        with open(json_file, 'rb') as f:
            client.get_blob_client(blob_name).upload_blob(f, overwrite=True)
        print(' done')
        uploaded.append(blob_name)

print(f'\nUploaded {len(uploaded)} result file(s) to blob storage.')
print('Download locally via Azure Storage Explorer or download_results_from_blob.py'
      ' → output/results/agentic/CRAG/')